# ERA5 Slice — SOCRATES/CAPRICORN Domain

Slices ERA5 pressure-level reanalysis from `/g/data/rt52/era5/pressure-levels/reanalysis/`  
to only what is needed for the EAE4024 group project.

**Domain**: Southern Ocean sector, Jan–Feb 2010–2025  
**Variables**: T, q, u, v, z (geopotential)  
**Output**: `era5_project.nc` — one file, ready for collocation

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr

# ── Verify we are on Gadi and the data root exists ──────────────────────────
ERA5_ROOT = '/g/data/rt52/era5/pressure-levels/reanalysis'
assert os.path.exists(ERA5_ROOT), (
    f'ERA5 root not found: {ERA5_ROOT}\n'
    'Run this notebook on Gadi (or mount the filesystem first).'
)
print('ERA5 root found:', ERA5_ROOT)
print('Variables available:', sorted(os.listdir(ERA5_ROOT)))

ERA5 root found: /g/data/rt52/era5/pressure-levels/reanalysis
Variables available: ['cc', 'ciwc', 'clwc', 'crwc', 'cswc', 'd', 'o3', 'pv', 'q', 'r', 't', 'u', 'v', 'vo', 'w', 'z']


In [5]:
# ── Project configuration ────────────────────────────────────────────────────

# Variables to extract  (folder name in rt52  →  output name in saved file)
VARIABLES = {
    't': 't',          # temperature (K)
    'q': 'q',          # specific humidity (kg/kg)
    'u': 'u',          # u-wind (m/s)
    'v': 'v',          # v-wind (m/s)
    'z': 'z',          # geopotential (m²/s²) → divide by 9.80665 for geopotential height
}

# Pressure levels (hPa) — surface to 500 hPa, good MABL resolution
PRESSURE_LEVELS = [1000, 975, 950, 925, 900, 875, 850, 825, 800,
                   775, 750, 700, 650, 600, 550, 500]

# Months
MONTHS = ['01', '02']  # January and February only

# Years:  2010–2025  (all Jan–Feb)
YEARS = list(range(2010, 2026))

# Spatial domain — covers SOCRATES/CAPRICORN cruise track + buffer
# ERA5 is on a 0.25° global grid; we select a box around the Southern Ocean sector
LAT_SLICE = slice(-45, -70)   # 45°S to 70°S  (xarray slice is inclusive)
LON_SLICE = slice(130, 175)   # 130°E to 175°E

OUTPUT_FILE = '/g/data/k10/zr7147/ERA5/PBL/era5_project.nc'

print(f'Years: {YEARS[0]}–{YEARS[-1]}, months: Jan–Feb')
print(f'Domain: {LAT_SLICE}, {LON_SLICE}')
print(f'Pressure levels: {PRESSURE_LEVELS}')

Years: 2010–2025, months: Jan–Feb
Domain: slice(-45, -70, None), slice(130, 175, None)
Pressure levels: [1000, 975, 950, 925, 900, 875, 850, 825, 800, 775, 750, 700, 650, 600, 550, 500]


In [4]:
# ── (Optional) Tighten domain from actual sounding locations ─────────────────
# If socrates_dropsondes.csv is in the same directory, shrink the box to
# the sounding track + 2° buffer.  Skip if file not present.

SONDE_CSV = 'socrates_dropsondes.csv'

if os.path.exists(SONDE_CSV):
    df_meta = pd.read_csv(SONDE_CSV)
    lats = df_meta['release_lat'].unique()
    lons = df_meta['release_lon'].unique()
    buf = 3.0  # degree buffer

    lat_min = np.floor(lats.min() - buf)
    lat_max = np.ceil(lats.max()  + buf)
    lon_min = np.floor(lons.min() - buf)
    lon_max = np.ceil(lons.max()  + buf)

    # ERA5 latitude is stored 90→-90 (descending), so slice order matters
    LAT_SLICE = slice(lat_max, lat_min)  # descending
    LON_SLICE = slice(lon_min, lon_max)

    print(f'Domain tightened from soundings: lat {lat_min}–{lat_max}, lon {lon_min}–{lon_max}')
else:
    print('socrates_dropsondes.csv not found — using default domain')
    # ERA5 lat is descending → flip slice
    LAT_SLICE = slice(-45, -70)
    LON_SLICE = slice(130, 175)

socrates_dropsondes.csv not found — using default domain


In [6]:
# ── Discover ERA5 files for all target years / months / variables ─────────────
#
# NCI rt52 file naming convention:
#   {ERA5_ROOT}/{var}/{year}/{var}_era5_oper_pl_{YYYYMM}01_{YYYYMM}{LD}.nc
# where LD = last day of month (28/29/30/31)

def find_era5_files(var, years, months):
    """Return sorted list of ERA5 monthly files for a variable."""
    files = []
    for year in years:
        for month in months:
            pattern = os.path.join(
                ERA5_ROOT, var, str(year),
                f'{var}_era5_oper_pl_{year}{month}*.nc'
            )
            found = sorted(glob.glob(pattern))
            if not found:
                print(f'  WARNING: no file found for {var} {year}-{month}')
            files.extend(found)
    return files

# Quick sanity check on temperature
t_files_2018 = find_era5_files('t', [2018], MONTHS)
print(f'Found {len(t_files_2018)} temperature file(s) for 2018:')
for f in t_files_2018:
    print(' ', f)

Found 2 temperature file(s) for 2018:
  /g/data/rt52/era5/pressure-levels/reanalysis/t/2018/t_era5_oper_pl_20180101-20180131.nc
  /g/data/rt52/era5/pressure-levels/reanalysis/t/2018/t_era5_oper_pl_20180201-20180228.nc


In [7]:
# ── Load, slice, and merge all variables ────────────────────────────────────
#
# Strategy: open each variable separately with open_mfdataset, slice the
# domain and pressure levels, then merge into one dataset.
# Using chunks for Dask-backed lazy loading (avoids RAM overrun).

ds_vars = {}

for var, out_name in VARIABLES.items():
    files = find_era5_files(var, YEARS, MONTHS)
    if not files:
        print(f'SKIP {var}: no files found')
        continue

    print(f'Opening {var}  ({len(files)} files) ...', end=' ', flush=True)

    ds = xr.open_mfdataset(
        files,
        combine='by_coords',
        chunks={'time': 4, 'level': 16},   # 4 time steps (1 day) per chunk
        engine='netcdf4',
    )

    # ERA5 uses 'level' for pressure (hPa), latitude descending, longitude 0→360
    ds = ds.sel(
        level=PRESSURE_LEVELS,
        latitude=LAT_SLICE,
        longitude=LON_SLICE,
    )

    # Keep only the data variable (drop any unexpected extras)
    ds_vars[out_name] = ds[[var]].rename({var: out_name})
    print('OK')

print('\nAll variables loaded.')

Opening t  (32 files) ... OK
Opening q  (32 files) ... OK
Opening u  (32 files) ... OK
Opening v  (32 files) ... OK
Opening z  (32 files) ... OK

All variables loaded.


In [8]:
# ── Merge into one dataset and inspect ──────────────────────────────────────

ds_era5 = xr.merge(list(ds_vars.values()))

# Add geopotential height as a convenience variable
if 'z' in ds_era5:
    ds_era5['z_height'] = ds_era5['z'] / 9.80665
    ds_era5['z_height'].attrs = {
        'long_name': 'Geopotential height',
        'units': 'm',
    }

# Global attributes
ds_era5.attrs.update({
    'project':     'EAE4024 SOCRATES/CAPRICORN ERA5 diagnostic',
    'source':      ERA5_ROOT,
    'created':     pd.Timestamp.now().isoformat(),
    'lat_domain':  str(LAT_SLICE),
    'lon_domain':  str(LON_SLICE),
    'years':       f'{YEARS[0]}–{YEARS[-1]}',
    'months':      'Jan–Feb',
})

print(ds_era5)
print(f'\nEstimated size in memory: {ds_era5.nbytes / 1e9:.2f} GB')

<xarray.Dataset> Size: 266GB
Dimensions:    (time: 22752, level: 16, latitude: 101, longitude: 181)
Coordinates:
  * time       (time) datetime64[ns] 182kB 2010-01-01 ... 2025-02-28T23:00:00
  * level      (level) int32 64B 1000 975 950 925 900 ... 700 650 600 550 500
  * latitude   (latitude) float32 404B -45.0 -45.25 -45.5 ... -69.5 -69.75 -70.0
  * longitude  (longitude) float32 724B 130.0 130.2 130.5 ... 174.5 174.8 175.0
Data variables:
    t          (time, level, latitude, longitude) float64 53GB dask.array<chunksize=(4, 12, 27, 20), meta=np.ndarray>
    q          (time, level, latitude, longitude) float64 53GB dask.array<chunksize=(4, 12, 27, 20), meta=np.ndarray>
    u          (time, level, latitude, longitude) float64 53GB dask.array<chunksize=(4, 12, 27, 20), meta=np.ndarray>
    v          (time, level, latitude, longitude) float64 53GB dask.array<chunksize=(4, 12, 27, 20), meta=np.ndarray>
    z          (time, level, latitude, longitude) float32 27GB dask.array<chunksiz

In [ ]:
# ── Quick sanity-check plot ──────────────────────────────────────────────────
# Mean 850 hPa temperature for January 2018 — should show Southern Ocean cold bias

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

mask = (ds_era5.time.dt.year == 2018) & (ds_era5.time.dt.month == 1)
t850_jan2018 = (
    ds_era5['t']
    .sel(level=850)
    .isel(time=mask)
    .mean('time')
    - 273.15  # K → °C
)

fig, ax = plt.subplots(
    figsize=(9, 5),
    subplot_kw={'projection': ccrs.PlateCarree()}
)
t850_jan2018.plot(
    ax=ax, transform=ccrs.PlateCarree(),
    cmap='RdBu_r', cbar_kwargs={'label': '°C'}
)
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=3)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=4)
ax.gridlines(draw_labels=True, alpha=0.3)
ax.set_title('ERA5 850 hPa Temperature — Jan 2018 mean (sanity check)')
plt.tight_layout()
plt.savefig('/tmp/era5_sanity_check.png', dpi=100)
plt.close()
print('Sanity-check plot saved to /tmp/era5_sanity_check.png')

In [ ]:
# ── Write to NetCDF ──────────────────────────────────────────────────────────

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)

# Remove any partial file from a previous failed run before writing
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)
    print(f'Removed existing partial file: {OUTPUT_FILE}')

# Rechunk so HDF5 chunk boundaries align exactly with Dask chunks.
# Misaligned chunks cause HDF5 read-modify-write on corrupt partial data → HDF error.
ds_write = ds_era5.chunk({'time': 4, 'level': 16, 'latitude': 101, 'longitude': 181})

encoding = {
    var: {
        'zlib': True,
        'complevel': 4,
        'dtype': 'float32',
        'chunksizes': (4, 16, 101, 181),
    }
    for var in ds_write.data_vars
}

print(f'Writing to {OUTPUT_FILE} ...')
ds_write.to_netcdf(OUTPUT_FILE, encoding=encoding)
size_mb = os.path.getsize(OUTPUT_FILE) / 1e6
print(f'Done. File size: {size_mb:.0f} MB')

In [ ]:
# ── Verify the saved file ────────────────────────────────────────────────────

ds_check = xr.open_dataset(OUTPUT_FILE)
print(ds_check)
print('\nTime range:', ds_check.time.values[0], '→', ds_check.time.values[-1])
print('Lat range: ', float(ds_check.latitude.min()), '→', float(ds_check.latitude.max()))
print('Lon range: ', float(ds_check.longitude.min()), '→', float(ds_check.longitude.max()))
print('Levels:    ', ds_check.level.values.tolist())
print('Variables: ', list(ds_check.data_vars))

In [ ]:
# ── Integrity check + plot verification ─────────────────────────────────────
# /g/data/k10/zr7147/ERA5/PBL/era5_project.nc — hardcoded, NCI path only

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import xarray as xr

CHECK_FILE = '/g/data/k10/zr7147/ERA5/PBL/era5_project.nc'
ds = xr.open_dataset(CHECK_FILE)

# ── 1. Structure ─────────────────────────────────────────────────────────────
print("── Structure ──")
assert ds.dims['time']      == 22752
assert ds.dims['level']     == 16
assert ds.dims['latitude']  == 101
assert ds.dims['longitude'] == 181
assert list(ds.data_vars)   == ['t', 'q', 'u', 'v', 'z', 'z_height']
print(f"  dims OK: {dict(ds.dims)}")
print(f"  vars OK: {list(ds.data_vars)}")

# ── 2. Time range ────────────────────────────────────────────────────────────
print("\n── Time range ──")
print(f"  {ds.time.values[0]} → {ds.time.values[-1]}")
assert str(ds.time.values[0])[:10] == '2010-01-01'
assert str(ds.time.values[-1])[:10] == '2025-02-28'
print("  OK ✓")

# ── 3. Physical range check (small slice to keep memory low) ─────────────────
print("\n── Physical ranges (first 4 timesteps) ──")
checks = {
    't':        (180,  320),
    'q':        (0,    0.03),
    'u':        (-80,  80),
    'v':        (-80,  80),
    'z':        (0,    2e5),
    'z_height': (0,    20000),
}
all_ok = True
for var, (lo, hi) in checks.items():
    s = ds[var].isel(time=slice(0, 4)).values
    nan_frac = np.isnan(s).mean()
    vmin, vmax = float(np.nanmin(s)), float(np.nanmax(s))
    ok = nan_frac == 0 and lo <= vmin and vmax <= hi
    all_ok = all_ok and ok
    flag = "✓" if ok else "✗ PROBLEM"
    print(f"  {var:8s}  [{vmin:10.4f}, {vmax:10.4f}]  NaN={nan_frac:.2%}  {flag}")

# ── 4. Six-panel plot (one per variable @ 850 hPa, Jan 2018 mean) ─────────────
print("\n── Generating 6-panel check plot ──")
mask = (ds.time.dt.year == 2018) & (ds.time.dt.month == 1)

panels = [
    ('t',        850, 'RdBu_r',  'T (K)'),
    ('q',        850, 'Blues',   'q (kg/kg)'),
    ('u',        850, 'RdBu_r',  'u (m/s)'),
    ('v',        850, 'RdBu_r',  'v (m/s)'),
    ('z_height', 850, 'viridis', 'z height (m)'),
    ('t',        500, 'RdBu_r',  'T 500 hPa (K)'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8),
                          subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (var, lev, cmap, label) in zip(axes.flat, panels):
    data = ds[var].sel(level=lev).isel(time=mask).mean('time')
    data.plot(ax=ax, transform=ccrs.PlateCarree(), cmap=cmap,
              cbar_kwargs={'label': label, 'shrink': 0.7})
    ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=3)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=4)
    ax.gridlines(alpha=0.3)
    ax.set_title(f'{var} @ {lev} hPa — Jan 2018')

plt.suptitle('ERA5 integrity check — Jan 2018 mean', fontsize=13)
plt.tight_layout()
out = '/tmp/era5_integrity_check.png'
plt.savefig(out, dpi=100, bbox_inches='tight')
plt.close()
print(f"  Saved → {out}")

print("\n" + "=" * 50)
print("ALL CHECKS PASSED ✓" if all_ok else "⚠ SOME CHECKS FAILED — review above")
print("=" * 50)
ds.close()

## Notes for running on Gadi

If the dataset is large (>~20 GB in memory), submit a PBS job instead of running interactively:

```bash
#!/bin/bash
#PBS -N era5_slice
#PBS -l ncpus=4
#PBS -l mem=32GB
#PBS -l walltime=02:00:00
#PBS -l storage=gdata/rt52
#PBS -l jobfs=10GB
#PBS -j oe

module load python3/3.11.7
source /path/to/your/venv/bin/activate

cd /path/to/this/notebook/directory
jupyter nbconvert --to notebook --execute era5_slice.ipynb --output era5_slice_run.ipynb
```

Key PBS flags:
- `#PBS -l storage=gdata/rt52` — **required** to access the ERA5 data on rt52
- `#PBS -l storage=gdata/rt52+scratch/xx` — add your own project storage if saving output outside `/scratch`